# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published on: {metadata.date_published}")
print(f"Cite as: {metadata.cite_as}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes tabular and non-tabular data into **Record Sets**, each described by a unique `@id`. Each **Record Set** contains **Fields**, which in turn correspond to columns in files and are also referenced by their unique `@id`s. We'll enumerate all available record sets, fields, and their structure.

In [ ]:
print("Available Record Sets:")
record_sets = []
for rs in metadata.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_sets.append(rs.id)

print("\nRecord Set Details:")
for rs in metadata.record_sets:
    print(f"\nRecordSet '@id': {rs.id}")
    print(f"Description: {rs.description}")
    print(f"Fields:")
    for f in rs.fields:
        col_str = f" (column '@id': {f.column.id})" if f.column else ""
        print(f" - {f.name} | field @id: {f.id}{col_str} | data type: {f.data_type}")

## 3. Data Extraction
Load table data from the chosen record set into a DataFrame for analysis. All tables and field names will be referenced by their Croissant `@id`s as listed above.

We'll extract all records from each table and display the columns available for analysis.

In [ ]:
dataframes = {}
# For demonstration, load all record sets into DataFrames, keyed by their `@id`s
for record_set_id in record_sets:
    print(f"\nLoading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set {record_set_id}:")
        print(df.columns.tolist())
        display(df.head())
    else:
        print(f"Warning: No records found for record set {record_set_id}.")    

## 4. Exploratory Data Analysis (EDA)
Let's process the main quantitative table. We'll:
- Select a numeric field (by its field `@id`) for filtering/normalization.
- Remove outliers (e.g., with values below a threshold).
- Perform simple normalization.
- Group by a key categorical field if available.

Replace `<main_record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with options listed above depending on your exploration needs.

In [ ]:
# Pick the first record set as main analysis set
# You can change this to the `@id` you'd like to work with
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print(main_df.columns)
    # Choose numeric fields (those with data type 'Number', 'Float', or 'Integer')
    # Let's attempt to find a numeric column
    field_map = {f.id: f for f in next(rs for rs in metadata.record_sets if rs.id == main_record_set_id).fields}
    numeric_fields = [f.id for f in field_map.values() if str(f.data_type).lower() in ["number", "float", "integer"]]
    if not numeric_fields:
        print("No numeric columns found for this record set.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Simple threshold: use mean as a splitter
        threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype != 'O' else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
        # Grouping: choose the first categorical field
        group_fields = [f.id for f in field_map.values() if str(f.data_type).lower() not in ["number", "float", "integer"]]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by '{group_field_id}':")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No record sets found in the metadata.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Use field `@id`s in all plots where possible for traceability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by a group field if possible
    if group_fields:
        plt.figure(figsize=(12,5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we have:
- Loaded Croissant metadata and explored available record sets and fields by their `@id`s.
- Loaded tabular data into Pandas DataFrames using `mlcroissant`.
- Performed initial exploratory analysis including basic filtering, normalization, and grouping by key fields.
- Visualized the distribution and relationships of quantitative fields using their canonical Croissant `@id`s.

**Next Steps:** You may conduct more in-depth domain-driven analysis based on the experimental context, combine data from different record sets, or export processed tables for downstream research.